## 6. Compute Metrics and Visualize Results

Aggregate evaluation metrics, plot comprehensive visualizations, and analyze labeling quality with multiple charts and summary tables.

In [ ]:
# Additional visualization: Create summary metrics table
print("\n" + "=" * 70)
print("SUMMARY METRICS TABLE")
print("=" * 70)

summary_data = {
    'Metric': [
        'Accuracy',
        'Macro Precision',
        'Macro Recall',
        'Macro F1-Score',
        'Weighted Precision',
        'Weighted Recall',
        'Weighted F1-Score',
        'Total Samples',
        'Correct Predictions',
        'Incorrect Predictions',
        'Error Rate (%)'
    ],
    'Value': [
        f'{accuracy:.4f}',
        f'{macro_precision:.4f}',
        f'{macro_recall:.4f}',
        f'{macro_f1:.4f}',
        f'{weighted_precision:.4f}',
        f'{weighted_recall:.4f}',
        f'{weighted_f1:.4f}',
        f'{len(ground_truth)}',
        f'{len(ground_truth) - errors}',
        f'{errors}',
        f'{error_rate*100:.2f}%'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + summary_df.to_string(index=False))

# Save results
print("\n" + "=" * 70)
print("RESULTS SAVED")
print("=" * 70)

# Save detailed predictions
results_df.to_csv('/tmp/adala_predictions.csv', index=False)
print("✓ Detailed predictions saved to: /tmp/adala_predictions.csv")

# Save metrics
metrics_export = {
    'accuracy': float(accuracy),
    'macro_precision': float(macro_precision),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
    'weighted_precision': float(weighted_precision),
    'weighted_recall': float(weighted_recall),
    'weighted_f1': float(weighted_f1),
    'total_samples': len(ground_truth),
    'correct_predictions': len(ground_truth) - errors,
    'incorrect_predictions': errors,
    'error_rate': float(error_rate),
    'per_class_metrics': {k: {kk: float(vv) for kk, vv in v.items()} 
                          for k, v in per_class_metrics.items()},
    'confusion_matrix': cm.tolist()
}

import json
with open('/tmp/adala_metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)
print("✓ Metrics saved to: /tmp/adala_metrics.json")

In [ ]:
# Section 6: Compute Metrics and Visualize Results

print("=" * 70)
print("VISUALIZATIONS")
print("=" * 70)

# 1. Confusion Matrix Heatmap
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Confusion Matrix
ax1 = axes[0, 0]
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax1)
ax1.set_title('Confusion Matrix - Sentiment Classification', fontsize=12, fontweight='bold')
ax1.set_ylabel('True Label')
ax1.set_xlabel('Predicted Label')

# Plot 2: Per-class metrics comparison
ax2 = axes[0, 1]
metrics_df = pd.DataFrame(per_class_metrics).T
metrics_df.plot(kind='bar', ax=ax2)
ax2.set_title('Per-Class Metrics Comparison', fontsize=12, fontweight='bold')
ax2.set_ylabel('Score')
ax2.set_xlabel('Sentiment Class')
ax2.legend(title='Metrics')
ax2.set_ylim([0, 1.1])
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Overall metrics
ax3 = axes[1, 0]
overall_metrics = {
    'Accuracy': accuracy,
    'Macro F1': macro_f1,
    'Weighted F1': weighted_f1
}
bars = ax3.bar(overall_metrics.keys(), overall_metrics.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax3.set_title('Overall Metrics', fontsize=12, fontweight='bold')
ax3.set_ylabel('Score')
ax3.set_ylim([0, 1.1])
for bar in bars:
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Predictions vs Ground Truth distribution
ax4 = axes[1, 1]
dist_data = pd.DataFrame({
    'Ground Truth': pd.Series(ground_truth).value_counts().sort_index(),
    'Predictions': pd.Series(predictions).value_counts().sort_index()
})
dist_data.plot(kind='bar', ax=ax4)
ax4.set_title('Label Distribution: Ground Truth vs Predictions', fontsize=12, fontweight='bold')
ax4.set_ylabel('Count')
ax4.set_xlabel('Sentiment Class')
ax4.legend(title='Source')
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Visualizations generated successfully!")

## 5. Evaluate Labeling Outputs

Calculate precision, recall, F1 score, confusion matrix, and error analysis for predicted labels.

In [ ]:
# Section 5: Evaluate Labeling Outputs

print("=" * 70)
print("EVALUATION METRICS - SENTIMENT LABELING")
print("=" * 70)

ground_truth = test_data['sentiment'].tolist()
predictions = predicted_sentiments
labels = ["Positive", "Negative", "Neutral"]

# Calculate basic metrics
accuracy = accuracy_score(ground_truth, predictions)
print(f"\n--- OVERALL METRICS ---")
print(f"Accuracy: {accuracy:.4f}")

# Macro-averaged metrics
macro_precision = precision_score(ground_truth, predictions, average='macro', zero_division=0)
macro_recall = recall_score(ground_truth, predictions, average='macro', zero_division=0)
macro_f1 = f1_score(ground_truth, predictions, average='macro', zero_division=0)

print(f"\nMacro-averaged (unweighted):")
print(f"  Precision: {macro_precision:.4f}")
print(f"  Recall:    {macro_recall:.4f}")
print(f"  F1-score:  {macro_f1:.4f}")

# Weighted metrics
weighted_precision = precision_score(ground_truth, predictions, average='weighted', zero_division=0)
weighted_recall = recall_score(ground_truth, predictions, average='weighted', zero_division=0)
weighted_f1 = f1_score(ground_truth, predictions, average='weighted', zero_division=0)

print(f"\nWeighted (by support):")
print(f"  Precision: {weighted_precision:.4f}")
print(f"  Recall:    {weighted_recall:.4f}")
print(f"  F1-score:  {weighted_f1:.4f}")

# Per-class metrics
print(f"\n--- PER-CLASS METRICS ---")
per_class_metrics = {}
for label in labels:
    y_true_binary = [1 if x == label else 0 for x in ground_truth]
    y_pred_binary = [1 if x == label else 0 for x in predictions]
    
    precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
    recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
    f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
    
    per_class_metrics[label] = {
        'precision': precision,
        'recall': recall,
        'f1': f1
    }
    
    print(f"\n{label}:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-score:  {f1:.4f}")

# Confusion matrix
print(f"\n--- CONFUSION MATRIX ---")
cm = confusion_matrix(ground_truth, predictions, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

# Classification report
print(f"\n--- SKLEARN CLASSIFICATION REPORT ---")
report = classification_report(ground_truth, predictions, labels=labels, zero_division=0)
print(report)

# Error analysis
errors = sum(1 for t, p in zip(ground_truth, predictions) if t != p)
error_rate = errors / len(ground_truth)
print(f"\n--- ERROR ANALYSIS ---")
print(f"Correct predictions: {len(ground_truth) - errors}")
print(f"Incorrect predictions: {errors}")
print(f"Error rate: {error_rate:.4f} ({error_rate*100:.2f}%)")

## 4. Generate Labels with Adala

Train the agent on ground truth data, then generate predictions for the test set.

In [ ]:
# Section 4: Generate Labels with Adala

print("=" * 70)
print("TRAINING ADALA AGENT")
print("=" * 70)
print("\nStarting agent training on ground truth data...")

# Train the agent
agent.learn()

print("\n✓ Agent training completed!")
print("\nAgent skills after training:")
print(agent.skills)

# Make predictions on test data
print("\n" + "=" * 70)
print("GENERATING PREDICTIONS")
print("=" * 70)
print("\nRunning predictions on test data...")

predictions_output = agent.run(test_data)

print("\n✓ Predictions completed!")
print(f"\nPredictions output type: {type(predictions_output)}")
print(f"Number of predictions: {len(predictions_output)}")

# Extract predicted sentiments
predicted_sentiments = []
for row in predictions_output:
    # Handle different possible output formats
    if isinstance(row, dict):
        predicted_sentiments.append(row.get('predicted_sentiment', 'Unknown'))
    else:
        predicted_sentiments.append(str(row))

# Create results dataframe
results_df = test_data.copy()
results_df['predicted_sentiment'] = predicted_sentiments
results_df['correct'] = results_df['sentiment'] == results_df['predicted_sentiment']

print("\n" + "─" * 70)
print("PREDICTION RESULTS")
print("─" * 70)
print(results_df.to_string(index=True))

## 3. Initialize Adala Labeling Pipeline

Configure and initialize the Adala agent with a sentiment classification skill.

In [ ]:
# Section 3: Initialize Adala Labeling Pipeline

# Import Adala components
from adala.agents import Agent
from adala.environments import StaticEnvironment
from adala.skills import ClassificationSkill

# Create the Adala agent with sentiment classification skill
agent = Agent(
    skills=ClassificationSkill(
        name='sentiment_classification',
        input_template='Review: {review}',
        output_template='Sentiment: {predicted_sentiment}',
        labels={
            'predicted_sentiment': [
                "Positive",
                "Negative",
                "Neutral"
            ]
        },
    ),
    environment=StaticEnvironment(
        df=training_data,
        ground_truth_columns={'predicted_sentiment': 'sentiment'}
    )
)

print("=" * 70)
print("ADALA AGENT INITIALIZED")
print("=" * 70)
print(f"\nAgent: {agent}")
print(f"\nSkills configured:\n{agent.skills}")

## 2. Load or Create Dataset

Create synthetic training and test datasets with customer reviews and sentiment labels.

In [ ]:
# Section 2: Load or Create Dataset
# Create training data with ground truth labels
training_data = pd.DataFrame([
    {"review": "Amazing product! Exceeded all expectations.", "sentiment": "Positive"},
    {"review": "Terrible quality. Very disappointed with my purchase.", "sentiment": "Negative"},
    {"review": "It's okay, nothing special.", "sentiment": "Neutral"},
    {"review": "Love it! Highly recommend to everyone.", "sentiment": "Positive"},
    {"review": "Waste of money. Complete disappointment.", "sentiment": "Negative"},
    {"review": "Good product, but could be better.", "sentiment": "Neutral"},
    {"review": "Absolutely fantastic! Worth every penny.", "sentiment": "Positive"},
    {"review": "Poor customer service and bad quality.", "sentiment": "Negative"},
    {"review": "Average product, does what it's supposed to.", "sentiment": "Neutral"},
    {"review": "Exceeded expectations! Very happy.", "sentiment": "Positive"},
    {"review": "Horrible experience. Never buying again.", "sentiment": "Negative"},
    {"review": "It works fine. Nothing to complain about.", "sentiment": "Neutral"},
    {"review": "Best purchase ever made!", "sentiment": "Positive"},
    {"review": "This is the worst product I've ever seen.", "sentiment": "Negative"},
    {"review": "Just okay, mid-range quality.", "sentiment": "Neutral"},
])

# Create test data for evaluation
test_data = pd.DataFrame([
    {"review": "Fantastic quality and fast delivery!", "sentiment": "Positive"},
    {"review": "Defective product upon arrival.", "sentiment": "Negative"},
    {"review": "Met expectations, satisfactory.", "sentiment": "Neutral"},
    {"review": "Outstanding! My family loves it.", "sentiment": "Positive"},
    {"review": "Not worth the price. Cheap materials.", "sentiment": "Negative"},
    {"review": "Okay product for the price.", "sentiment": "Neutral"},
    {"review": "I love this! Perfect for my needs.", "sentiment": "Positive"},
    {"review": "Broke after one week. Very disappointed.", "sentiment": "Negative"},
    {"review": "It's a regular product. Does the job.", "sentiment": "Neutral"},
    {"review": "Excellent service and quality!", "sentiment": "Positive"},
])

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)
print(f"\nTraining data shape: {training_data.shape}")
print(f"Test data shape: {test_data.shape}")
print(f"\nTraining data:\n{training_data}")
print(f"\nTest data:\n{test_data}")

# Display class distribution
print("\n" + "─" * 70)
print("CLASS DISTRIBUTION")
print("─" * 70)
print("\nTraining data:")
print(training_data['sentiment'].value_counts().sort_index())
print("\nTest data:")
print(test_data['sentiment'].value_counts().sort_index())

## 1. Import Required Libraries

Import Adala and supporting libraries for data handling, labeling, and evaluation.

In [ ]:
# Section 1: Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

# Adala Sentiment Analysis Labeling with Evaluation Metrics

This notebook demonstrates how to:
1. Use **Adala** for autonomous data labeling
2. Generate sentiment labels for customer reviews
3. Evaluate labeling accuracy with comprehensive metrics
4. Visualize results and analyze labeling quality

**Task**: Classify customer reviews into sentiment categories (Positive, Negative, Neutral)